In [ ]:

# --- 1. FILE PATHS ---
var_char_file = 'Var_char_Fmin_Tese_Varactor_characterization_tb2_Measurements_history_1_20260416_16_39_25.18.csv'
var_real_file = 'TOTAL_VTEMP_REAL_CORNERS_LC_NEWVAR_F_PrimeSim_default_VtempReal_Measurements_history_1_20260615_19_17_31.60.csv'

# --- 2. UNIT PARSING ---
unit_map = {'f': 1e-15, 'p': 1e-12, 'n': 1e-9, 'u': 1e-6, 'm': 1e-3, 'k': 1e3, 'M': 1e6, 'G': 1e9}

def parse_units(value):
    if pd.isna(value) or str(value).strip().lower() == "error" or str(value).strip() == "":
        return np.nan
    val_str = str(value).strip()
    match = re.search(r'([0-9\.-]+)([a-zA-Z]*)', val_str)
    if match:
        num = float(match.group(1))
        multiplier = unit_map.get(match.group(2), 1.0)
        return num * multiplier
    return np.nan

# --- 3. LOAD & PREPARE DATA ---
df_char = pd.read_csv(var_char_file)
df_real = pd.read_csv(var_real_file)

# Parse uncompensated varactor data (char)
df_char['Cap_fF'] = df_char['Cap_measuredAt1Mhz_M7:ac'].apply(parse_units) * 1e15
df_char['Q'] = pd.to_numeric(df_char['Q_factorAt1Mhz:ac'], errors='coerce')
df_char['Temp'] = pd.to_numeric(df_char['Sweep:Variable:temp'], errors='coerce')

# Split Char Corners (Uncompensated)
df_char_tt = df_char[df_char['Corner'] == 'TT25'].sort_values('Temp')
df_char_ss = df_char[df_char['Corner'] == 'SS125'].sort_values('Temp')
df_char_ff = df_char[df_char['Corner'] == 'FF-40'].sort_values('Temp')

# Parse real compensated varactor data
df_real['Cap_fF'] = df_real['Cap_max_measured:ac'].apply(parse_units) * 1e15
df_real['Q'] = pd.to_numeric(df_real['Q_factor:ac'], errors='coerce')
df_real['Temp'] = pd.to_numeric(df_real['Corner:Variable:temp'], errors='coerce')

# Split Real Corners (Compensated)
df_real_tt = df_real[df_real['Corner'].str.startswith('TT')].sort_values('Temp')
df_real_ss = df_real[df_real['Corner'].str.startswith('SS')].sort_values('Temp')
df_real_ff = df_real[df_real['Corner'].str.startswith('FF')].sort_values('Temp')

# --- 4. PLOT COMPARISONS ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))

# Capacitance vs Temperature (CT)
# Real Compensated (Solid Lines)
ax1.plot(df_real_tt['Temp'], df_real_tt['Cap_fF'], label='TT Compensated (Real Vtemp)', marker='o', color='red', linewidth=2)
ax1.plot(df_real_ss['Temp'], df_real_ss['Cap_fF'], label='SS Compensated (Real Vtemp)', marker='o', color='blue', linewidth=2)
ax1.plot(df_real_ff['Temp'], df_real_ff['Cap_fF'], label='FF Compensated (Real Vtemp)', marker='o', color='green', linewidth=2)

# Uncompensated (Dashed Lines)
ax1.plot(df_char_tt['Temp'], df_char_tt['Cap_fF'], label='TT Uncompensated', marker='s', linestyle='--', color='red', alpha=0.6, linewidth=2)
ax1.plot(df_char_ss['Temp'], df_char_ss['Cap_fF'], label='SS Uncompensated', marker='s', linestyle='--', color='blue', alpha=0.6, linewidth=2)
ax1.plot(df_char_ff['Temp'], df_char_ff['Cap_fF'], label='FF Uncompensated', marker='s', linestyle='--', color='green', alpha=0.6, linewidth=2)

ax1.axhline(181.0, color='gray', linestyle=':', linewidth=2, alpha=0.7, label='Target (181 fF)')
ax1.set_xlabel('Temperature (°C)')
ax1.set_ylabel('Capacitance (fF)')
ax1.set_title('CT Curve Comparison (All Corners)', fontweight='bold')
ax1.legend()
ax1.grid(True, linestyle='--', alpha=0.7)

# Quality Factor vs Temperature
# Real Compensated (Solid Lines)
ax2.plot(df_real_tt['Temp'], df_real_tt['Q'], label='TT Compensated (Real Vtemp)', marker='o', color='red', linewidth=2)
ax2.plot(df_real_ss['Temp'], df_real_ss['Q'], label='SS Compensated (Real Vtemp)', marker='o', color='blue', linewidth=2)
ax2.plot(df_real_ff['Temp'], df_real_ff['Q'], label='FF Compensated (Real Vtemp)', marker='o', color='green', linewidth=2)

# Uncompensated (Dashed Lines)
ax2.plot(df_char_tt['Temp'], df_char_tt['Q'], label='TT Uncompensated', marker='s', linestyle='--', color='red', alpha=0.6, linewidth=2)
ax2.plot(df_char_ss['Temp'], df_char_ss['Q'], label='SS Uncompensated', marker='s', linestyle='--', color='blue', alpha=0.6, linewidth=2)
ax2.plot(df_char_ff['Temp'], df_char_ff['Q'], label='FF Uncompensated', marker='s', linestyle='--', color='green', alpha=0.6, linewidth=2)

ax2.set_xlabel('Temperature (°C)')
ax2.set_ylabel('Q Factor')
ax2.set_title('Q Factor Comparison (All Corners)', fontweight='bold')
ax2.legend()
ax2.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

# --- 5. CALCULATE PEAK-TO-PEAK PERCENTAGE ERROR ---

target_cap = 181.0  # Standardization baseline (fF)

def calc_err(df):
    if df.empty: return 0.0
    stand = df['Cap_fF'] / target_cap
    peak_to_peak_err = (stand.max() - stand.min()) * 100
    
    # Check overall slope direction (Highest Temp vs Lowest Temp)
    # If the curve goes down as temp goes up, apply a negative sign.
    direction = -1 if stand.iloc[-1] < stand.iloc[0] else 1
    return peak_to_peak_err * direction

print("\n--- STANDARDIZED CAPACITANCE PEAK-TO-PEAK ERROR SUMMARY ---")
print(f"Target Baseline used for Standardization: {target_cap:.2f} fF\n")

print("1. COMPENSATED CURVES (REAL VTEMP)")
print(f"   TT Corner Total Percentual Error (Directional): {calc_err(df_real_tt):.2f}%")
print(f"   SS Corner Total Percentual Error (Directional): {calc_err(df_real_ss):.2f}%")
print(f"   FF Corner Total Percentual Error (Directional): {calc_err(df_real_ff):.2f}%\n")

print("2. UNCOMPENSATED CURVES")
print(f"   TT Corner Total Percentual Error (Directional): {calc_err(df_char_tt):.2f}%")
print(f"   SS Corner Total Percentual Error (Directional): {calc_err(df_char_ss):.2f}%")
print(f"   FF Corner Total Percentual Error (Directional): {calc_err(df_char_ff):.2f}%\n")